In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
df_train = pd.read_csv("/content/DailyDelhiClimateTest.csv", parse_dates=['date'])
df_test =pd.read_csv("/content/DailyDelhiClimateTest.csv", parse_dates=['date'])

In [4]:
df_train.head()


,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


In [5]:
df_test.head()


,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


In [6]:
df_train.describe()

,date,meantemp,humidity,wind_speed,meanpressure
count,114,114.000000,114.000000,114.000000,114.000000
mean,2017-02-26 12:00:00,21.713079,56.258362,8.143924,1004.035090
min,2017-01-01 00:00:00,11.000000,17.750000,1.387500,59.000000
25%,2017-01-29 06:00:00,16.437198,39.625000,5.563542,1007.437500
50%,2017-02-26 12:00:00,19.875000,57.750000,8.069444,1012.739316
75%,2017-03-26 18:00:00,27.705357,71.902778,10.068750,1016.739583
max,2017-04-24 00:00:00,34.500000,95.833333,19.314286,1022.809524
std,NaN,6.360072,19.068083,3.588049,89.474692


In [9]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          114 non-null    datetime64[ns]
 1   meantemp      114 non-null    float64       
 2   humidity      114 non-null    float64       
 3   wind_speed    114 non-null    float64       
 4   meanpressure  114 non-null    float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 4.6 KB


In [10]:
date_train=df_train['date']
date_test=df_test['date']

In [11]:
df_train = df_train.set_index('date')
df_test=df_test.set_index('date')

In [12]:
df_train.sample()

,meantemp,humidity,wind_speed,meanpressure
date,,,,
2017-03-11,18.533333,60.4,5.566667,1009.8


In [13]:
df_test.sample()

,meantemp,humidity,wind_speed,meanpressure
date,,,,
2017-03-20,23.333333,54.666667,10.077778,1012.555556


In [14]:
df_train['year'] = df_train.index.year
df_train['month'] = df_train.index.month
df_train['day_name'] = df_train.index.strftime('%A')
df_train

,meantemp,humidity,wind_speed,meanpressure,year,month,day_name
date,,,,,,,
2017-01-01,15.913043,85.869565,2.743478,59.000000,2017,1,Sunday
2017-01-02,18.500000,77.222222,2.894444,1018.277778,2017,1,Monday
2017-01-03,17.111111,81.888889,4.016667,1018.333333,2017,1,Tuesday
2017-01-04,18.700000,70.050000,4.545000,1015.700000,2017,1,Wednesday
2017-01-05,18.388889,74.944444,3.300000,1014.333333,2017,1,Thursday
...,...,...,...,...,...,...,...
2017-04-20,34.500000,27.500000,5.562500,998.625000,2017,4,Thursday
2017-04-21,34.250000,39.375000,6.962500,999.875000,2017,4,Friday
2017-04-22,32.900000,40.900000,8.890000,1001.600000,2017,4,Saturday


In [15]:
df_test['year'] = df_test.index.year
df_test['month'] = df_test.index.month
df_test['day_name'] = df_test.index.strftime('%A')
df_test


,meantemp,humidity,wind_speed,meanpressure,year,month,day_name
date,,,,,,,
2017-01-01,15.913043,85.869565,2.743478,59.000000,2017,1,Sunday
2017-01-02,18.500000,77.222222,2.894444,1018.277778,2017,1,Monday
2017-01-03,17.111111,81.888889,4.016667,1018.333333,2017,1,Tuesday
2017-01-04,18.700000,70.050000,4.545000,1015.700000,2017,1,Wednesday
2017-01-05,18.388889,74.944444,3.300000,1014.333333,2017,1,Thursday
...,...,...,...,...,...,...,...
2017-04-20,34.500000,27.500000,5.562500,998.625000,2017,4,Thursday
2017-04-21,34.250000,39.375000,6.962500,999.875000,2017,4,Friday
2017-04-22,32.900000,40.900000,8.890000,1001.600000,2017,4,Saturday


In [20]:
df_train.loc['2017']

,meantemp,humidity,wind_speed,meanpressure,year,month,day_name
date,,,,,,,
2017-01-01,15.913043,85.869565,2.743478,59.000000,2017,1,Sunday
2017-01-02,18.500000,77.222222,2.894444,1018.277778,2017,1,Monday
2017-01-03,17.111111,81.888889,4.016667,1018.333333,2017,1,Tuesday
2017-01-04,18.700000,70.050000,4.545000,1015.700000,2017,1,Wednesday
2017-01-05,18.388889,74.944444,3.300000,1014.333333,2017,1,Thursday
...,...,...,...,...,...,...,...
2017-04-20,34.500000,27.500000,5.562500,998.625000,2017,4,Thursday
2017-04-21,34.250000,39.375000,6.962500,999.875000,2017,4,Friday
2017-04-22,32.900000,40.900000,8.890000,1001.600000,2017,4,Saturday


In [21]:
df_train.loc['2017-01-01':'2017-02-01']

,meantemp,humidity,wind_speed,meanpressure,year,month,day_name
date,,,,,,,
2017-01-01,15.913043,85.869565,2.743478,59.000000,2017,1,Sunday
2017-01-02,18.500000,77.222222,2.894444,1018.277778,2017,1,Monday
2017-01-03,17.111111,81.888889,4.016667,1018.333333,2017,1,Tuesday
2017-01-04,18.700000,70.050000,4.545000,1015.700000,2017,1,Wednesday
2017-01-05,18.388889,74.944444,3.300000,1014.333333,2017,1,Thursday
2017-01-06,19.318182,79.318182,8.681818,1011.772727,2017,1,Friday
2017-01-07,14.708333,95.833333,10.041667,1011.375000,2017,1,Saturday
2017-01-08,15.684211,83.526316,1.950000,1015.550000,2017,1,Sunday
2017-01-09,14.571429,80.809524,6.542857,1015.952381,2017,1,Monday


In [25]:
# Mean Temperature
fig_temp = go.Figure()
fig_temp.add_trace(go.Scatter(x=df_train.index, y=df_train['meantemp'], mode='lines'))
fig_temp.update_layout(title="Mean Temperature", xaxis_title="Date", yaxis_title="Value", template='plotly_white', title_x=0.5)
fig_temp.show()



In [26]:
# Humidity
fig_humidity = go.Figure()
fig_humidity.add_trace(go.Scatter(x=df_train.index, y=df_train['humidity'], mode='lines'))
fig_humidity.update_layout(title="Humidity", xaxis_title="Date", yaxis_title="Value", template='plotly_white', title_x=0.5)
fig_humidity.show()

In [28]:
# Wind Speed
fig_wind = go.Figure()
fig_wind.add_trace(go.Scatter(x=df_train.index, y=df_train['wind_speed'], mode='lines'))
fig_wind.update_layout(title="Wind Speed", xaxis_title="Date", yaxis_title="Value", template='plotly_white', title_x=0.5)
fig_wind.show()



In [29]:
# Mean Pressure
fig_pressure = go.Figure()
fig_pressure.add_trace(go.Scatter(x=df_train.index, y=df_train['meanpressure'], mode='lines'))
fig_pressure.update_layout(title="Mean Pressure", xaxis_title="Date", yaxis_title="Value", template='plotly_white', title_x=0.5)
fig_pressure.show()

In [30]:
df_train.dtypes

,0
meantemp,float64
humidity,float64
wind_speed,float64
meanpressure,float64
year,int32
month,int32
day_name,object


In [31]:
df_train_Numerical = df_train.drop(columns=['day_name'])
df_test_Numerical = df_test.drop(columns=['day_name'])

In [33]:
df_train_Numerical.columns


Index(['meantemp', 'humidity', 'wind_speed', 'meanpressure', 'year', 'month'], dtype='object')

In [34]:
df_test_Numerical.columns

Index(['meantemp', 'humidity', 'wind_speed', 'meanpressure', 'year', 'month'], dtype='object')

In [35]:
scl = StandardScaler()
df_train_scl = scl.fit_transform(df_train_Numerical)
df_test_scl = scl.transform(df_test_Numerical)

In [36]:
type(df_train_scl)

numpy.ndarray

In [37]:
df_train_scl=pd.DataFrame(df_train_scl)
df_train_scl

,0,1,2,3,4,5
0,-0.915971,1.559776,-1.511765,-10.608670,0.0,-1.292571
1,-0.507426,1.104275,-1.469505,0.159884,0.0,-1.292571
2,-0.726766,1.350093,-1.155357,0.160508,0.0,-1.292571
3,-0.475841,0.726477,-1.007459,0.130947,0.0,-1.292571
4,-0.524973,0.984293,-1.355976,0.115605,0.0,-1.292571
...,...,...,...,...,...,...
109,2.019376,-1.514852,-0.722627,-0.060732,0.0,1.436191
110,1.979895,-0.889334,-0.330720,-0.046700,0.0,1.436191
111,1.766696,-0.809005,0.208852,-0.027336,0.0,1.436191
112,1.762748,-1.514852,0.509080,-0.021442,0.0,1.436191


In [38]:
df_test_scl=pd.DataFrame(df_test_scl)
df_test_scl

,0,1,2,3,4,5
0,-0.915971,1.559776,-1.511765,-10.608670,0.0,-1.292571
1,-0.507426,1.104275,-1.469505,0.159884,0.0,-1.292571
2,-0.726766,1.350093,-1.155357,0.160508,0.0,-1.292571
3,-0.475841,0.726477,-1.007459,0.130947,0.0,-1.292571
4,-0.524973,0.984293,-1.355976,0.115605,0.0,-1.292571
...,...,...,...,...,...,...
109,2.019376,-1.514852,-0.722627,-0.060732,0.0,1.436191
110,1.979895,-0.889334,-0.330720,-0.046700,0.0,1.436191
111,1.766696,-0.809005,0.208852,-0.027336,0.0,1.436191
112,1.762748,-1.514852,0.509080,-0.021442,0.0,1.436191


In [39]:
df_train_scl.rename(columns={0: 'meantemp', 1: 'humidity',2:'wind_speed',3:'meanpressure',4:'year',5:'month'}, inplace=True)
df_train_scl

,meantemp,humidity,wind_speed,meanpressure,year,month
0,-0.915971,1.559776,-1.511765,-10.608670,0.0,-1.292571
1,-0.507426,1.104275,-1.469505,0.159884,0.0,-1.292571
2,-0.726766,1.350093,-1.155357,0.160508,0.0,-1.292571
3,-0.475841,0.726477,-1.007459,0.130947,0.0,-1.292571
4,-0.524973,0.984293,-1.355976,0.115605,0.0,-1.292571
...,...,...,...,...,...,...
109,2.019376,-1.514852,-0.722627,-0.060732,0.0,1.436191
110,1.979895,-0.889334,-0.330720,-0.046700,0.0,1.436191
111,1.766696,-0.809005,0.208852,-0.027336,0.0,1.436191
112,1.762748,-1.514852,0.509080,-0.021442,0.0,1.436191


In [40]:
df_test_scl.rename(columns={0: 'meantemp', 1: 'humidity',2:'wind_speed',3:'meanpressure',4:'year',5:'month'}, inplace=True)
df_test_scl

,meantemp,humidity,wind_speed,meanpressure,year,month
0,-0.915971,1.559776,-1.511765,-10.608670,0.0,-1.292571
1,-0.507426,1.104275,-1.469505,0.159884,0.0,-1.292571
2,-0.726766,1.350093,-1.155357,0.160508,0.0,-1.292571
3,-0.475841,0.726477,-1.007459,0.130947,0.0,-1.292571
4,-0.524973,0.984293,-1.355976,0.115605,0.0,-1.292571
...,...,...,...,...,...,...
109,2.019376,-1.514852,-0.722627,-0.060732,0.0,1.436191
110,1.979895,-0.889334,-0.330720,-0.046700,0.0,1.436191
111,1.766696,-0.809005,0.208852,-0.027336,0.0,1.436191
112,1.762748,-1.514852,0.509080,-0.021442,0.0,1.436191


In [41]:
df_date_train = pd.DataFrame({'date_train': date_train}) #to df
df_train_scl = pd.concat([df_train_scl, df_date_train], axis=1)#concat
print(df_train_scl.columns)

Index(['meantemp', 'humidity', 'wind_speed', 'meanpressure', 'year', 'month',
       'date_train'],
      dtype='object')


In [42]:
df_train_scl.set_index('date_train', inplace=True)
print(df_train_scl)


            meantemp  humidity  wind_speed  meanpressure  year     month
date_train                                                              
2017-01-01 -0.915971  1.559776   -1.511765    -10.608670   0.0 -1.292571
2017-01-02 -0.507426  1.104275   -1.469505      0.159884   0.0 -1.292571
2017-01-03 -0.726766  1.350093   -1.155357      0.160508   0.0 -1.292571
2017-01-04 -0.475841  0.726477   -1.007459      0.130947   0.0 -1.292571
2017-01-05 -0.524973  0.984293   -1.355976      0.115605   0.0 -1.292571
...              ...       ...         ...           ...   ...       ...
2017-04-20  2.019376 -1.514852   -0.722627     -0.060732   0.0  1.436191
2017-04-21  1.979895 -0.889334   -0.330720     -0.046700   0.0  1.436191
2017-04-22  1.766696 -0.809005    0.208852     -0.027336   0.0  1.436191
2017-04-23  1.762748 -1.514852    0.509080     -0.021442   0.0  1.436191
2017-04-24  1.624563 -1.533665    1.123434      0.001210   0.0  1.436191

[114 rows x 6 columns]


In [43]:
df_date_test = pd.DataFrame({'date_test': date_test})
df_test_scl = pd.concat([df_test_scl, df_date_test], axis=1)
print(df_test_scl.columns)


Index(['meantemp', 'humidity', 'wind_speed', 'meanpressure', 'year', 'month',
       'date_test'],
      dtype='object')


In [44]:
df_test_scl.set_index('date_test', inplace=True)
print(df_test_scl)

            meantemp  humidity  wind_speed  meanpressure  year     month
date_test                                                               
2017-01-01 -0.915971  1.559776   -1.511765    -10.608670   0.0 -1.292571
2017-01-02 -0.507426  1.104275   -1.469505      0.159884   0.0 -1.292571
2017-01-03 -0.726766  1.350093   -1.155357      0.160508   0.0 -1.292571
2017-01-04 -0.475841  0.726477   -1.007459      0.130947   0.0 -1.292571
2017-01-05 -0.524973  0.984293   -1.355976      0.115605   0.0 -1.292571
...              ...       ...         ...           ...   ...       ...
2017-04-20  2.019376 -1.514852   -0.722627     -0.060732   0.0  1.436191
2017-04-21  1.979895 -0.889334   -0.330720     -0.046700   0.0  1.436191
2017-04-22  1.766696 -0.809005    0.208852     -0.027336   0.0  1.436191
2017-04-23  1.762748 -1.514852    0.509080     -0.021442   0.0  1.436191
2017-04-24  1.624563 -1.533665    1.123434      0.001210   0.0  1.436191

[114 rows x 6 columns]


Extract the endogenous variable(wind speed)

In [45]:
wind_train = df_train_scl['wind_speed']
wind_test = df_test_scl['wind_speed']

In [60]:
#Train the AR mode
from statsmodels.tsa.ar_model import AutoReg

wind_train.index = pd.to_datetime(wind_train.index)


wind_train = wind_train.asfreq('D')

if len(wind_train) > lag_order:
    # Fit the AutoReg model
    lag_order = 180
    ar_model_wind = AutoReg(wind_train, lags=lag_order)
    model_fit = ar_model_wind.fit()
    print(model_fit.summary())
else:
    print(f"Data has fewer than {lag_order} observations. Adjust lag_order.")

Data has fewer than 180 observations. Adjust lag_order.


In [63]:
predictions = model_fit.predict(start=len(wind_train), end=len(wind_train)+len(wind_test)-1, dynamic=False)
print(predictions)


114      NaN
115      NaN
116      NaN
117      NaN
118      NaN
       ...  
223    225.0
224    228.0
225    225.0
226    228.0
227    231.0
Length: 114, dtype: float64


In [62]:
mse_wind = mean_squared_error(wind_test, predictions)
print(f'Mean Squared Error: {mse_wind:.5f}')

ValueError: Input contains NaN.